# The 2025 Federal Workforce Shock

In [10]:
import duckdb
import pathlib
import numpy as np
import pandas as pd
import pmdarima as pm

Matplotlib is building the font cache; this may take a moment.


# P1: Shock Overview:
1. Data Engineering & Preparation  Python (DuckDB)
2. Econometric & Statistical Modeling Python (pmdarima / statsmodels)
3. Data Model & Semantic Layer Power BI (Power Query)
4. Dashboard Canvas & UI Build (Power Query)

## 1. Data Engineering & Preparation

### 1.1 Separation Data Ingestion & Sanitization

In [20]:
con = duckdb.connect()

print("=" * 60)
print("CHECK 1: Row Weight / Count Check")
print("=" * 60)
# Testing on December 2024 employment
check1 = con.sql("""
    SELECT 
        MIN(CAST(count AS INT)) AS min_count,
        MAX(CAST(count AS INT)) AS max_count,
        COUNT(*) AS total_rows,
        SUM(CAST(count AS INT)) AS total_people
    FROM read_csv_auto('data/employment/employment_202412_*.txt', all_varchar=true)
""").df()
print(check1)

print("\n" + "=" * 60)
print("CHECKS 2 & 3: DRP Indicator & Separation Categories")
print("=" * 60)
# Look across all 2025 separation files for DRP activity
check2_3 = con.sql("""
    SELECT 
        drp_indicator, 
        separation_category, 
        SUM(CAST(count AS INT)) AS n
    FROM read_csv_auto('data/separations/separations_2025*.txt', all_varchar=true)
    GROUP BY ALL 
    ORDER BY n DESC
    LIMIT 10
""").df()
print(check2_3)

print("\n" + "=" * 60)
print("CHECK 5: Pay Null Rate (GS Full-Time in Dec 2024)")
print("=" * 60)
check5 = con.sql("""
    SELECT 
        COUNT(*) AS gs_ft_rows,
        SUM(CASE WHEN annualized_adjusted_basic_pay IS NULL OR annualized_adjusted_basic_pay = '' THEN 1 ELSE 0 END) AS null_pay_count,
        AVG(CASE WHEN annualized_adjusted_basic_pay IS NULL OR annualized_adjusted_basic_pay = '' THEN 1.0 ELSE 0.0 END) AS pay_null_rate
    FROM read_csv_auto('data/employment/employment_202412_*.txt', all_varchar=true)
    WHERE pay_plan_code = 'GS'
      AND work_schedule_code = 'F'
""").df()
print(check5)

CHECK 1: Row Weight / Count Check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   min_count  max_count  total_rows  total_people
0          1          1     2312301     2312301.0

CHECKS 2 & 3: DRP Indicator & Separation Categories
  drp_indicator                 separation_category        n
0             N                                QUIT  91618.0
1             Y                                QUIT  64748.0
2             N              RETIREMENT - VOLUNTARY  57596.0
3             Y              RETIREMENT - VOLUNTARY  40449.0
4             N    TERMINATION (EXPIRED APPT/OTHER)  35308.0
5             N                    OTHER SEPARATION  15745.0
6             Y              RETIREMENT - EARLY OUT  14354.0
7             N  TRANSFER OUT - INDIVIDUAL TRANSFER  10670.0
8             N            REDUCTION IN FORCE (RIF)  10136.0
9             N              RETIREMENT - EARLY OUT   9264.0

CHECK 5: Pay Null Rate (GS Full-Time in Dec 2024)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   gs_ft_rows  null_pay_count  pay_null_rate
0     1481990            43.0       0.000029


### 1.3 DRP & Organic Time-Series Aggregation

In [21]:
con = duckdb.connect("opm.duckdb")

for ds in ["separations", "accessions", "employment"]:
    out = pathlib.Path(f"parquet/{ds}")
    out.mkdir(parents=True, exist_ok=True)
    
    raw_files = sorted(list(pathlib.Path(f"data/{ds}").glob("*.txt")) + 
                       list(pathlib.Path(f"data/{ds}").glob("*.csv")))
    
    print(f"Converting {len(raw_files)} files in data/{ds}...")
    
    for f in raw_files:
        dest_file = out / (f.stem + ".parquet")
        if not dest_file.exists():
            con.execute(f"""
                COPY (
                    SELECT *, '{f.name}' AS source_file
                    FROM read_csv_auto('{f.as_posix()}', all_varchar = true)
                )
                TO '{dest_file.as_posix()}' (FORMAT parquet)
            """)

print("All datasets successfully converted to Parquet.")

Converting 32 files in data/separations...
Converting 32 files in data/accessions...
Converting 32 files in data/employment...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

All datasets successfully converted to Parquet.


In [34]:
# -------------------------------------------------------------
# WBS 1.3 - 1. Separations View (v_sep)
# Filtered for yyyymm >= 202312 to eliminate retroactive records
# -------------------------------------------------------------
con.execute("""
CREATE OR REPLACE VIEW v_sep AS
SELECT
    CAST(personnel_action_effective_date_yyyymm AS INT)       AS yyyymm,
    department_code, 
    agency_code, 
    agency,
    occupational_category_code                                AS occ_cat, 
    occupational_group_code                                   AS occ_group,
    pay_plan_code, 
    grade,
    TRY_CAST(length_of_service_years AS DOUBLE)               AS los,
    age_bracket, 
    supervisory_status_code                                   AS sup, 
    tenure_code                                               AS tenure,
    veteran_indicator                                         AS veteran,
    separation_category_code                                  AS sep_cat, 
    drp_indicator                                             AS drp_flag,
    TRY_CAST(annualized_adjusted_basic_pay AS DOUBLE)         AS pay,
    CAST(count AS INT)                                        AS n,
    CASE 
        WHEN TRY_CAST(length_of_service_years AS DOUBLE) < 3  THEN '00-02' 
        WHEN TRY_CAST(length_of_service_years AS DOUBLE) < 6  THEN '03-05'
        WHEN TRY_CAST(length_of_service_years AS DOUBLE) < 11 THEN '06-10' 
        WHEN TRY_CAST(length_of_service_years AS DOUBLE) < 21 THEN '11-20' 
        ELSE '21+' 
    END AS los_band,
    CASE 
        WHEN pay_plan_code <> 'GS' THEN 'Other pay plan'
        WHEN TRY_CAST(grade AS INT) <= 7  THEN 'GS 01-07'
        WHEN TRY_CAST(grade AS INT) <= 11 THEN 'GS 08-11'
        WHEN TRY_CAST(grade AS INT) <= 13 THEN 'GS 12-13'
        ELSE 'GS 14-15' 
    END AS grade_band
FROM read_parquet('parquet/separations/*.parquet')
WHERE CAST(personnel_action_effective_date_yyyymm AS INT) >= 202312;
""")

# -------------------------------------------------------------
# WBS 1.3 - 2. Accessions View (v_acc)
# Filtered for yyyymm >= 202312
# -------------------------------------------------------------
con.execute("""
CREATE OR REPLACE VIEW v_acc AS
SELECT
    CAST(personnel_action_effective_date_yyyymm AS INT)       AS yyyymm,
    department_code, 
    agency_code, 
    agency,
    occupational_category_code                                AS occ_cat, 
    occupational_group_code                                   AS occ_group,
    pay_plan_code, 
    grade,
    TRY_CAST(length_of_service_years AS DOUBLE)               AS los,
    age_bracket, 
    accession_category_code                                   AS acc_cat,
    TRY_CAST(annualized_adjusted_basic_pay AS DOUBLE)         AS pay,
    CAST(count AS INT)                                        AS n,
    CASE 
        WHEN TRY_CAST(length_of_service_years AS DOUBLE) < 3  THEN '00-02' 
        WHEN TRY_CAST(length_of_service_years AS DOUBLE) < 6  THEN '03-05'
        WHEN TRY_CAST(length_of_service_years AS DOUBLE) < 11 THEN '06-10' 
        WHEN TRY_CAST(length_of_service_years AS DOUBLE) < 21 THEN '11-20' 
        ELSE '21+' 
    END AS los_band,
    CASE 
        WHEN pay_plan_code <> 'GS' THEN 'Other pay plan'
        WHEN TRY_CAST(grade AS INT) <= 7  THEN 'GS 01-07'
        WHEN TRY_CAST(grade AS INT) <= 11 THEN 'GS 08-11'
        WHEN TRY_CAST(grade AS INT) <= 13 THEN 'GS 12-13'
        ELSE 'GS 14-15' 
    END AS grade_band
FROM read_parquet('parquet/accessions/*.parquet')
WHERE CAST(personnel_action_effective_date_yyyymm AS INT) >= 202312;
""")

# -------------------------------------------------------------
# WBS 1.3 - 3. Employment View (v_emp)
# -------------------------------------------------------------
con.execute("""
CREATE OR REPLACE VIEW v_emp AS
SELECT
    CAST(snapshot_yyyymm AS INT)                               AS yyyymm,
    department_code, 
    agency_code, 
    agency,
    occupational_category_code                                AS occ_cat, 
    occupational_group_code                                   AS occ_group,
    pay_plan_code, 
    grade,
    locality_pay_area_code                                    AS loc, 
    work_schedule_code                                        AS work_sched,
    TRY_CAST(length_of_service_years AS DOUBLE)               AS los,
    age_bracket, 
    supervisory_status_code                                   AS sup, 
    tenure_code                                               AS tenure,
    veteran_indicator                                         AS veteran,
    TRY_CAST(annualized_adjusted_basic_pay AS DOUBLE)         AS pay,
    CAST(count AS INT)                                        AS n,
    CASE 
        WHEN TRY_CAST(length_of_service_years AS DOUBLE) < 3  THEN '00-02' 
        WHEN TRY_CAST(length_of_service_years AS DOUBLE) < 6  THEN '03-05'
        WHEN TRY_CAST(length_of_service_years AS DOUBLE) < 11 THEN '06-10' 
        WHEN TRY_CAST(length_of_service_years AS DOUBLE) < 21 THEN '11-20' 
        ELSE '21+' 
    END AS los_band,
    CASE 
        WHEN pay_plan_code <> 'GS' THEN 'Other pay plan'
        WHEN TRY_CAST(grade AS INT) <= 7  THEN 'GS 01-07'
        WHEN TRY_CAST(grade AS INT) <= 11 THEN 'GS 08-11'
        WHEN TRY_CAST(grade AS INT) <= 13 THEN 'GS 12-13'
        ELSE 'GS 14-15' 
    END AS grade_band
FROM read_parquet('parquet/employment/*.parquet');
""")

# -------------------------------------------------------------
# WBS 1.3 - 4. Aggregated Headcount Fact Table
# -------------------------------------------------------------
con.execute("""
CREATE OR REPLACE TABLE fact_headcount AS
SELECT 
    yyyymm, 
    department_code, 
    agency_code,
    occ_cat, 
    grade_band, 
    los_band, 
    age_bracket, 
    SUM(n) AS headcount
FROM v_emp
GROUP BY ALL;
""")

print("All views and fact_headcount materialized with proper date filters.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

All views and fact_headcount materialized with proper date filters.


### 1.4 Top-10 Agency Baseline Slicing

In [9]:
con = duckdb.connect("opm.duckdb")

# WBS 1.2: Monthly separations aggregation
print("--- Aggregating Monthly Separations (WBS 1.2) ---")
df_ts = con.sql("""
    SELECT 
        yyyymm,
        SUM(n) AS total,
        SUM(CASE WHEN drp_flag = 'Y' THEN n ELSE 0 END) AS drp,
        SUM(CASE WHEN drp_flag <> 'Y' OR drp_flag IS NULL THEN n ELSE 0 END) AS organic
    FROM v_sep
    GROUP BY 1
    ORDER BY 1
""").df()

df_ts["month"] = pd.to_datetime(df_ts["yyyymm"].astype(str), format="%Y%m")
print(df_ts.tail(8))

# WBS 1.3: Identify Top 10 Agencies by Baseline Headcount (Dec 2024) from v_emp
print("\n--- Top 10 Agencies by Dec-2024 Headcount (WBS 1.3) ---")
top10_agencies = con.sql("""
    SELECT 
        agency_code, 
        agency, 
        SUM(n) AS baseline_headcount
    FROM v_emp
    WHERE yyyymm = 202412
    GROUP BY 1, 2
    ORDER BY baseline_headcount DESC
    LIMIT 10
""").df()
print(top10_agencies)

--- Aggregating Monthly Separations (WBS 1.2) ---
    yyyymm     total       drp  organic      month
39  202505   26762.0     456.0  26306.0 2025-05-01
40  202506   20145.0     511.0  19634.0 2025-06-01
41  202507   23241.0     491.0  22750.0 2025-07-01
42  202508   19257.0    1025.0  18232.0 2025-08-01
43  202509  124958.0  100117.0  24841.0 2025-09-01
44  202510   16900.0    2547.0  14353.0 2025-10-01
45  202511   13152.0     533.0  12619.0 2025-11-01
46  202512   25609.0   14101.0  11508.0 2025-12-01

--- Top 10 Agencies by Dec-2024 Headcount (WBS 1.3) ---
  agency_code                                   agency  baseline_headcount
0          VA           DEPARTMENT OF VETERANS AFFAIRS            480075.0
1          HS          DEPARTMENT OF HOMELAND SECURITY            231337.0
2          AR                   DEPARTMENT OF THE ARMY            222831.0
3          NV                   DEPARTMENT OF THE NAVY            221918.0
4          AF              DEPARTMENT OF THE AIR FORCE     

## 2. Econometric & Statistical Modeling

### 2.1 Pre-Shock Baseline Back-Testing

Train a SARIMA model on pre-shock history (e.g., through Dec-2023), forecast 2024 actuals, and compute baseline MAPE to prove model validity.

In [27]:
# -------------------------------------------------------------
# WBS 2.1: Monthly counts inspection to confirm continuous baseline
# -------------------------------------------------------------
df_check_counts = con.sql("""
    SELECT yyyymm, SUM(n) AS separations
    FROM v_sep
    GROUP BY 1
    ORDER BY 1
""").df()

print("Monthly Separations Volume (Pre-Shock Inspection):")
print(df_check_counts.head(15))

# -------------------------------------------------------------
# WBS 2.2: Counterfactual Shock Forecasting (Dec-2023 to Jan-2025 baseline)
# -------------------------------------------------------------
# Filter clean continuous series starting from December 2023
df_clean_ts = df_ts[df_ts["yyyymm"] >= 202312].copy()
s_model = df_clean_ts.set_index("month")["total"].asfreq("MS").interpolate(method="time")

# Define pre-shock training window (Dec 2023 through Jan 2025)
y_train_pre = s_model[:"2025-01"]
y_shock_actual = s_model["2025-02":]

print(f"\nTraining pre-shock model on {len(y_train_pre)} months...")

# Fit ARIMA baseline model for counterfactual projection
cf_model = pm.auto_arima(
    y_train_pre,
    seasonal=False,
    with_intercept=True,
    trend="c",
    suppress_warnings=True,
    error_action="ignore"
)

# Forecast counterfactual horizon with 95% confidence intervals
pred_cf, conf_int = cf_model.predict(
    n_periods=len(y_shock_actual), 
    return_conf_int=True, 
    alpha=0.05
)

# Compile results table for shock evaluation
df_cf_eval = pd.DataFrame({
    "actual": y_shock_actual.values,
    "forecast": np.round(pred_cf.values, 0),
    "ci_lower": np.round(conf_int[:, 0], 0),
    "ci_upper": np.round(conf_int[:, 1], 0),
    "excess": np.round(y_shock_actual.values - pred_cf.values, 0)
}, index=y_shock_actual.index.strftime("%Y-%m"))

print("\n=== WBS 2.2 DELIVERABLE: COUNTERFACTUAL & SHOCK IMPACT ===")
print(df_cf_eval.head(10))

Monthly Separations Volume (Pre-Shock Inspection):
    yyyymm  separations
0   202201          1.0
1   202203          4.0
2   202204          7.0
3   202205          9.0
4   202206        113.0
5   202207        123.0
6   202208        134.0
7   202209        190.0
8   202210         92.0
9   202211         79.0
10  202212        142.0
11  202301        110.0
12  202302        130.0
13  202303        148.0
14  202304        165.0

Training pre-shock model on 14 months...

=== WBS 2.2 DELIVERABLE: COUNTERFACTUAL & SHOCK IMPACT ===
           actual  forecast  ci_lower  ci_upper    excess
month                                                    
2025-02   15695.0   19464.0   13702.0   25225.0   -3769.0
2025-03   22308.0   19543.0   12903.0   26183.0    2765.0
2025-04   24405.0   19543.0   12903.0   26183.0    4862.0
2025-05   27281.0   19543.0   12903.0   26183.0    7738.0
2025-06   20540.0   19543.0   12903.0   26183.0     997.0
2025-07   23841.0   19543.0   12903.0   26183.0    4298.0

In [28]:
# -------------------------------------------------------------
# WBS 2.3: Generate Counterfactual Estimates for Total + Top 10 Agencies
# -------------------------------------------------------------
# Fetch list of Top 10 agencies by baseline headcount
top10_list = con.sql("""
    SELECT agency_code 
    FROM v_emp 
    WHERE yyyymm = 202412 
    GROUP BY 1 
    ORDER BY SUM(n) DESC 
    LIMIT 10
""").df()["agency_code"].tolist()

# Define function to run baseline fit and project counterfactual horizon
def model_series(df_slice, agency_id="TOTAL"):
    s_full = df_slice.set_index("month")["total"].asfreq("MS").interpolate(method="time")
    
    # Pre-shock window and shock horizon
    y_pre = s_full[:"2025-01"]
    y_shock = s_full["2025-02":]
    
    # Fit baseline model
    model = pm.auto_arima(
        y_pre,
        seasonal=False,
        with_intercept=True,
        trend="c",
        suppress_warnings=True,
        error_action="ignore"
    )
    
    # Forecast horizon with 95% confidence interval
    pred_vals, ci = model.predict(n_periods=len(y_shock), return_conf_int=True, alpha=0.05)
    
    # Prepare structured dataframe
    df_out = pd.DataFrame({
        "month": y_shock.index,
        "yyyymm": y_shock.index.strftime("%Y%m").astype(int),
        "agency_code": agency_id,
        "actual": y_shock.values,
        "forecast": np.round(pred_vals.values, 0),
        "ci_lower": np.round(ci[:, 0], 0),
        "ci_upper": np.round(ci[:, 1], 0),
        "excess": np.round(y_shock.values - pred_vals.values, 0)
    })
    
    # Merge DRP attributed separations
    df_drp = df_slice[["month", "drp"]].rename(columns={"drp": "drp_separations"})
    df_out = df_out.merge(df_drp, on="month", how="left").fillna(0)
    
    return df_out

forecast_frames = []

# 1. Estimate for Total Federal Workforce
print("Modeling total federal separations...")
df_total_ts = con.sql("""
    SELECT 
        yyyymm,
        SUM(n) AS total,
        SUM(CASE WHEN drp_flag = 'Y' THEN n ELSE 0 END) AS drp
    FROM v_sep
    WHERE yyyymm >= 202312
    GROUP BY 1
    ORDER BY 1
""").df()
df_total_ts["month"] = pd.to_datetime(df_total_ts["yyyymm"].astype(str), format="%Y%m")
forecast_frames.append(model_series(df_total_ts, agency_id="TOTAL"))

# 2. Estimate for each Top 10 agency
for ag in top10_list:
    print(f"Modeling agency: {ag}...")
    df_ag_ts = con.sql(f"""
        SELECT 
            yyyymm,
            SUM(n) AS total,
            SUM(CASE WHEN drp_flag = 'Y' THEN n ELSE 0 END) AS drp
        FROM v_sep
        WHERE yyyymm >= 202312 AND agency_code = '{ag}'
        GROUP BY 1
        ORDER BY 1
    """).df()
    df_ag_ts["month"] = pd.to_datetime(df_ag_ts["yyyymm"].astype(str), format="%Y%m")
    forecast_frames.append(model_series(df_ag_ts, agency_id=ag))

# Concatenate all forecasts into single fact table
fct_forecast = pd.concat(forecast_frames, ignore_index=True)

# -------------------------------------------------------------
# WBS 2.4: Materialize Forecast Fact Table to Parquet
# -------------------------------------------------------------
out_dir = pathlib.Path("model_outputs")
out_dir.mkdir(parents=True, exist_ok=True)
output_path = out_dir / "fct_forecast.parquet"

fct_forecast.to_parquet(output_path, index=False)

print(f"\n=== WBS 2.4 DELIVERABLE: FACT FORECAST EXPORTED ===")
print(f"Saved to: {output_path}")
print(f"Total rows: {len(fct_forecast)}")
print(f"Distinct agencies modeled: {fct_forecast['agency_code'].nunique()}")
print("\nSample records (Top 5 rows of materialized table):")
print(fct_forecast.head(5))

Modeling total federal separations...
Modeling agency: VA...
Modeling agency: HS...
Modeling agency: AR...
Modeling agency: NV...
Modeling agency: AF...
Modeling agency: DD...
Modeling agency: DJ...
Modeling agency: TR...
Modeling agency: HE...
Modeling agency: AG...

=== WBS 2.4 DELIVERABLE: FACT FORECAST EXPORTED ===
Saved to: model_outputs\fct_forecast.parquet
Total rows: 198
Distinct agencies modeled: 11

Sample records (Top 5 rows of materialized table):
       month  yyyymm agency_code   actual  forecast  ci_lower  ci_upper  \
0 2025-02-01  202502       TOTAL  15695.0   19464.0   13702.0   25225.0   
1 2025-03-01  202503       TOTAL  22308.0   19543.0   12903.0   26183.0   
2 2025-04-01  202504       TOTAL  24405.0   19543.0   12903.0   26183.0   
3 2025-05-01  202505       TOTAL  27281.0   19543.0   12903.0   26183.0   
4 2025-06-01  202506       TOTAL  20540.0   19543.0   12903.0   26183.0   

   excess  drp_separations  
0 -3769.0             42.0  
1  2765.0            152.0 

In [29]:
# -------------------------------------------------------------
# WBS 2.5: Grouped Logistic Regression (Segment Risk Analysis)
# -------------------------------------------------------------
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Step 1: Aggregate denominator and DRP exits by segment in 2025
df_segments = con.sql("""
    WITH baseline_emp AS (
        SELECT 
            occ_cat,
            grade_band,
            los_band,
            SUM(n) AS total_headcount
        FROM v_emp
        WHERE yyyymm = 202412
        GROUP BY 1, 2, 3
    ),
    drp_exits AS (
        SELECT 
            occ_cat,
            grade_band,
            los_band,
            SUM(n) AS drp_exits
        FROM v_sep
        WHERE yyyymm BETWEEN 202501 AND 202512
          AND drp_flag = 'Y'
        GROUP BY 1, 2, 3
    )
    SELECT 
        b.occ_cat,
        b.grade_band,
        b.los_band,
        b.total_headcount,
        COALESCE(e.drp_exits, 0) AS drp_exits,
        COALESCE(e.drp_exits, 0) * 1.0 / b.total_headcount AS observed_rate
    FROM baseline_emp b
    LEFT JOIN drp_exits e 
      ON b.occ_cat = e.occ_cat 
     AND b.grade_band = e.grade_band 
     AND b.los_band = e.los_band
    WHERE b.total_headcount >= 30
""").df()

# Step 2: Fit Grouped Binomial GLM (Logistic link)
df_segments["non_drp"] = df_segments["total_headcount"] - df_segments["drp_exits"]
df_segments["non_drp"] = df_segments["non_drp"].clip(lower=0)

formula = "drp_exits + non_drp ~ C(occ_cat) + C(grade_band) + C(los_band)"
glm_model = smf.glm(
    formula=formula,
    data=df_segments,
    family=sm.families.Binomial()
).fit()

# Step 3: Compute model predicted probabilities and risk tiers
df_segments["predicted_rate"] = glm_model.predict(df_segments)

# Assign risk tiers based on predicted departure rate percentiles
p33 = df_segments["predicted_rate"].quantile(0.33)
p66 = df_segments["predicted_rate"].quantile(0.66)

df_segments["risk_tier"] = np.where(
    df_segments["predicted_rate"] >= p66, "High Risk",
    np.where(df_segments["predicted_rate"] >= p33, "Medium Risk", "Low Risk")
)

# Step 4: Materialize segment risk output for Power BI (P2)
output_risk_path = out_dir / "fct_segment_risk.parquet"
df_segments.to_parquet(output_risk_path, index=False)

print("=== WBS 2.5 DELIVERABLE: SEGMENT RISK TABLE MATERIALIZED ===")
print(f"Saved to: {output_risk_path}")
print(f"Total segment cells analyzed: {len(df_segments)}")
print("\nTop 5 Highest Risk Segments Identified:")
print(
    df_segments.sort_values(by="predicted_rate", ascending=False)[
        ["occ_cat", "grade_band", "los_band", "total_headcount", "drp_exits", "predicted_rate", "risk_tier"]
    ].head(5)
)

=== WBS 2.5 DELIVERABLE: SEGMENT RISK TABLE MATERIALIZED ===
Saved to: model_outputs\fct_segment_risk.parquet
Total segment cells analyzed: 139

Top 5 Highest Risk Segments Identified:
   occ_cat grade_band los_band  total_headcount  drp_exits  predicted_rate  \
54       C   GS 14-15      21+            308.0       96.0        0.054157   
25       P   GS 14-15      21+          27671.0     1611.0        0.052656   
83       A   GS 14-15      21+          42829.0     1621.0        0.043453   
0        C   GS 12-13      21+            602.0      125.0        0.034415   
82       P   GS 12-13      21+          31613.0     1299.0        0.033442   

    risk_tier  
54  High Risk  
25  High Risk  
83  High Risk  
0   High Risk  
82  High Risk  


In [30]:
# -------------------------------------------------------------
# WBS 2.6: Pay Elasticity and Salary Band Exit Analysis
# -------------------------------------------------------------
# Step 1: Query salary distribution, headcount, and DRP departures
df_pay = con.sql("""
    WITH baseline_pay AS (
        SELECT 
            agency_code,
            occ_cat,
            grade_band,
            FLOOR(pay / 10000) * 10000 AS salary_band_min,
            COUNT(*) AS total_headcount,
            AVG(pay) AS avg_salary
        FROM v_emp
        WHERE yyyymm = 202412 AND pay > 20000 AND pay < 300000
        GROUP BY 1, 2, 3, 4
    ),
    drp_pay_exits AS (
        SELECT 
            agency_code,
            occ_cat,
            grade_band,
            FLOOR(pay / 10000) * 10000 AS salary_band_min,
            COUNT(*) AS drp_exits
        FROM v_sep
        WHERE yyyymm BETWEEN 202501 AND 202512 
          AND drp_flag = 'Y'
          AND pay > 20000 AND pay < 300000
        GROUP BY 1, 2, 3, 4
    )
    SELECT 
        b.agency_code,
        b.occ_cat,
        b.grade_band,
        CAST(b.salary_band_min AS INT) AS salary_band_min,
        CAST(b.salary_band_min + 10000 AS INT) AS salary_band_max,
        b.avg_salary,
        b.total_headcount,
        COALESCE(e.drp_exits, 0) AS drp_exits,
        COALESCE(e.drp_exits, 0) * 1.0 / b.total_headcount AS exit_rate
    FROM baseline_pay b
    LEFT JOIN drp_pay_exits e 
      ON b.agency_code = e.agency_code
     AND b.occ_cat = e.occ_cat
     AND b.grade_band = e.grade_band
     AND b.salary_band_min = e.salary_band_min
    WHERE b.total_headcount >= 20
""").df()

# Step 2: Fit OLS Log-Pay model to quantify elasticity
df_pay["log_salary"] = np.log(df_pay["avg_salary"])
ols_model = smf.wls(
    "exit_rate ~ log_salary + C(occ_cat) + C(grade_band)",
    data=df_pay,
    weights=df_pay["total_headcount"]
).fit()

# Calculate modeled/fitted exit rate based on salary
df_pay["modeled_exit_rate"] = ols_model.predict(df_pay).clip(lower=0)

# Step 3: Materialize Pay Elasticity Output for Power BI (P3)
output_pay_path = out_dir / "fct_pay_elasticity.parquet"
df_pay.to_parquet(output_pay_path, index=False)

print("=== WBS 2.6 DELIVERABLE: PAY ELASTICITY TABLE MATERIALIZED ===")
print(f"Saved to: {output_pay_path}")
print(f"Rows aggregated across pay bands: {len(df_pay)}")
print(f"Log-Pay Coefficient (Elasticity slope): {ols_model.params['log_salary']:.5f} (p-value: {ols_model.pvalues['log_salary']:.4e})")

print("\nSummary of Exit Rates by Salary Tier:")
summary_by_tier = df_pay.groupby("salary_band_min").agg(
    total_headcount=("total_headcount", "sum"),
    drp_exits=("drp_exits", "sum"),
    avg_exit_rate=("exit_rate", "mean"),
    modeled_exit_rate=("modeled_exit_rate", "mean")
).reset_index()

print(summary_by_tier.head(8))

=== WBS 2.6 DELIVERABLE: PAY ELASTICITY TABLE MATERIALIZED ===
Saved to: model_outputs\fct_pay_elasticity.parquet
Rows aggregated across pay bands: 2480
Log-Pay Coefficient (Elasticity slope): 0.01668 (p-value: 1.5687e-07)

Summary of Exit Rates by Salary Tier:
   salary_band_min  total_headcount  drp_exits  avg_exit_rate  \
0            20000              114          0       0.000000   
1            30000            25052         39       0.007219   
2            40000            82371        415       0.016754   
3            50000           110912        660       0.016614   
4            60000           101339        779       0.018045   
5            70000            94785        813       0.019827   
6            80000            85964        850       0.022610   
7            90000            86524        946       0.024102   

   modeled_exit_rate  
0           0.000000  
1           0.001151  
2           0.002811  
3           0.005056  
4           0.007113  
5           0.

In [31]:
# -------------------------------------------------------------
# WBS 2.7: Agency Clustering and Archetypes Classification
# -------------------------------------------------------------
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Step 1: Compute multivariate agency-level shock metrics
df_agency_metrics = con.sql("""
    WITH baseline AS (
        SELECT 
            agency_code,
            agency,
            SUM(n) AS baseline_headcount
        FROM v_emp
        WHERE yyyymm = 202412
        GROUP BY 1, 2
        HAVING SUM(n) >= 500
    ),
    shock_exits AS (
        SELECT 
            agency_code,
            SUM(CASE WHEN drp_flag = 'Y' THEN n ELSE 0 END) AS drp_exits,
            SUM(CASE WHEN drp_flag = 'Y' AND (los_band IN ('11-20', '21+') OR grade_band IN ('GS 12-13', 'GS 14-15')) THEN n ELSE 0 END) AS senior_drp_exits,
            SUM(CASE WHEN drp_flag = 'Y' AND sup = '2' THEN n ELSE 0 END) AS sup_drp_exits
        FROM v_sep
        WHERE yyyymm BETWEEN 202501 AND 202512
        GROUP BY 1
    )
    SELECT 
        b.agency_code,
        b.agency,
        b.baseline_headcount,
        COALESCE(s.drp_exits, 0) AS drp_exits,
        COALESCE(s.drp_exits, 0) * 1.0 / b.baseline_headcount AS drp_exit_rate,
        COALESCE(s.senior_drp_exits, 0) * 1.0 / NULLIF(s.drp_exits, 0) AS senior_exit_ratio,
        COALESCE(s.sup_drp_exits, 0) * 1.0 / b.baseline_headcount AS supervisor_exit_rate
    FROM baseline b
    LEFT JOIN shock_exits s ON b.agency_code = s.agency_code
""").df()

# Fill missing ratios for agencies with 0 exits
df_agency_metrics["senior_exit_ratio"] = df_agency_metrics["senior_exit_ratio"].fillna(0)

# Step 2: Scale metrics and run K-Means
cluster_features = ["drp_exit_rate", "senior_exit_ratio", "supervisor_exit_rate"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_agency_metrics[cluster_features])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_agency_metrics["cluster_id"] = kmeans.fit_predict(X_scaled)

# Step 3: Map cluster IDs to business archetypes based on centroid properties
centroids = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_), 
    columns=cluster_features
)

# Label clusters dynamically based on rank of exit rates and seniority
high_rate_idx = centroids["drp_exit_rate"].idxmax()
high_senior_idx = centroids.drop(index=high_rate_idx)["senior_exit_ratio"].idxmax()
low_impact_idx = centroids["drp_exit_rate"].idxmin()
remaining_idx = [i for i in range(4) if i not in [high_rate_idx, high_senior_idx, low_impact_idx]][0]

archetype_labels = {
    high_rate_idx: "Mass Workforce Contraction",
    high_senior_idx: "Severe Brain Drain",
    low_impact_idx: "Protected / Low Impact",
    remaining_idx: "Selective Management Shift"
}

df_agency_metrics["archetype"] = df_agency_metrics["cluster_id"].map(archetype_labels)

# Step 4: Materialize Archetypes Dimension for Power BI (P4)
output_archetype_path = out_dir / "dim_agency_archetype.parquet"
df_agency_metrics.to_parquet(output_archetype_path, index=False)

print("=== WBS 2.7 DELIVERABLE: AGENCY ARCHETYPES MATERIALIZED ===")
print(f"Saved to: {output_archetype_path}")
print(f"Agencies classified: {len(df_agency_metrics)}")
print("\nCluster Profiles (Centroid Characteristics):")
print(centroids)

print("\nDistribution of Agencies by Archetype:")
print(df_agency_metrics["archetype"].value_counts())

print("\nSample Agencies in Severe Brain Drain & Mass Contraction:")
print(
    df_agency_metrics[df_agency_metrics["archetype"].isin(["Severe Brain Drain", "Mass Workforce Contraction"])]
    [["agency_code", "agency", "baseline_headcount", "drp_exit_rate", "archetype"]]
    .head(6)
)

=== WBS 2.7 DELIVERABLE: AGENCY ARCHETYPES MATERIALIZED ===
Saved to: model_outputs\dim_agency_archetype.parquet
Agencies classified: 49

Cluster Profiles (Centroid Characteristics):
   drp_exit_rate  senior_exit_ratio  supervisor_exit_rate
0       0.045855           0.810036              0.007259
1       0.265333           0.200536              0.046614
2       0.120877           0.401147              0.010667
3       0.207881           0.860708              0.034538

Distribution of Agencies by Archetype:
archetype
Protected / Low Impact        34
Severe Brain Drain             8
Selective Management Shift     5
Mass Workforce Contraction     2
Name: count, dtype: int64

Sample Agencies in Severe Brain Drain & Mass Contraction:
   agency_code                                    agency  baseline_headcount  \
1           EP           ENVIRONMENTAL PROTECTION AGENCY             16990.0   
5           CU      NATIONAL CREDIT UNION ADMINISTRATION              1229.0   
12          OM      

In [35]:
df_refill = con.sql("""
    WITH monthly_acc AS (
        SELECT 
            yyyymm,
            agency_code,
            agency,
            occ_cat,
            SUM(n) AS accessions
        FROM v_acc
        WHERE yyyymm >= 202401
        GROUP BY 1, 2, 3, 4
    ),
    monthly_sep AS (
        SELECT 
            yyyymm,
            agency_code,
            agency,
            occ_cat,
            SUM(n) AS separations,
            SUM(CASE WHEN drp_flag = 'Y' THEN n ELSE 0 END) AS drp_separations
        FROM v_sep
        WHERE yyyymm >= 202401
        GROUP BY 1, 2, 3, 4
    )
    SELECT 
        COALESCE(s.yyyymm, a.yyyymm) AS yyyymm,
        COALESCE(s.agency_code, a.agency_code) AS agency_code,
        COALESCE(s.agency, a.agency) AS agency,
        COALESCE(s.occ_cat, a.occ_cat) AS occ_cat,
        COALESCE(a.accessions, 0) AS accessions,
        COALESCE(s.separations, 0) AS separations,
        COALESCE(s.drp_separations, 0) AS drp_separations,
        COALESCE(a.accessions, 0) - COALESCE(s.separations, 0) AS net_flow,
        ROUND(COALESCE(a.accessions, 0) * 1.0 / NULLIF(COALESCE(s.separations, 0), 0), 3) AS hire_to_exit_ratio
    FROM monthly_sep s
    FULL OUTER JOIN monthly_acc a 
      ON s.yyyymm = a.yyyymm 
     AND s.agency_code = a.agency_code 
     AND s.occ_cat = a.occ_cat
    ORDER BY 1, 2
""").df()

# Add standard datetime column
df_refill["month"] = pd.to_datetime(df_refill["yyyymm"].astype(str), format="%Y%m")

# Materialize Refill Gap table for Power BI (P5)
output_refill_path = out_dir / "fct_refill_gap.parquet"
df_refill.to_parquet(output_refill_path, index=False)

print("=== WBS 2.8 DELIVERABLE: REFILL GAP FACT TABLE MATERIALIZED ===")
print(f"Saved to: {output_refill_path}")
print(f"Total grain records: {len(df_refill)}")

# Print summary of net deficit during 2025 shock window
summary_2025 = df_refill[df_refill["yyyymm"].between(202501, 202512)].agg({
    "accessions": "sum",
    "separations": "sum",
    "drp_separations": "sum",
    "net_flow": "sum"
})

print("\nFull Year 2025 Total Workforce Flow Summary:")
print(f"Total Hires (Accessions):        {summary_2025['accessions']:,.0f}")
print(f"Total Exits (Separations):       {summary_2025['separations']:,.0f}")
print(f"Of which DRP Exits:              {summary_2025['drp_separations']:,.0f}")
print(f"Net Deficit (Net Flow):          {summary_2025['net_flow']:,.0f}")
print(f"Overall 2025 Hire-to-Exit Ratio: {summary_2025['accessions'] / summary_2025['separations']:.2f}")

=== WBS 2.8 DELIVERABLE: REFILL GAP FACT TABLE MATERIALIZED ===
Saved to: model_outputs\fct_refill_gap.parquet
Total grain records: 8593

Full Year 2025 Total Workforce Flow Summary:
Total Hires (Accessions):        127,693
Total Exits (Separations):       391,433
Of which DRP Exits:              138,174
Net Deficit (Net Flow):          -263,740
Overall 2025 Hire-to-Exit Ratio: 0.33


## 3. Data Model & Semantic Layer Power BI (Power Query)

### 3.1: Materialize Star Schema Core Dimensions & Facts

In [36]:
# -------------------------------------------------------------
# WBS 3.1: Materialize Star Schema Core Dimensions & Facts
# -------------------------------------------------------------
export_dir = pathlib.Path("export")
export_dir.mkdir(parents=True, exist_ok=True)

# 1. Dimension: Agency
con.execute("""
CREATE OR REPLACE TABLE dim_agency AS
SELECT DISTINCT 
    agency_code,
    department_code,
    agency
FROM (
    SELECT agency_code, department_code, agency FROM v_emp
    UNION
    SELECT agency_code, department_code, agency FROM v_sep
    UNION
    SELECT agency_code, department_code, agency FROM v_acc
);
""")
con.execute(f"COPY dim_agency TO '{(export_dir / 'dim_agency.parquet').as_posix()}' (FORMAT parquet);")
print("Exported: dim_agency.parquet")

# 2. Dimension: Date (Monthly Calendar)
con.execute("""
CREATE OR REPLACE TABLE dim_date AS
WITH distinct_months AS (
    SELECT DISTINCT yyyymm FROM v_sep
    UNION
    SELECT DISTINCT yyyymm FROM v_emp
    UNION
    SELECT DISTINCT yyyymm FROM v_acc
)
SELECT 
    yyyymm,
    STRPTIME(CAST(yyyymm AS VARCHAR), '%Y%m') AS date_ms,
    YEAR(STRPTIME(CAST(yyyymm AS VARCHAR), '%Y%m')) AS cal_year,
    MONTH(STRPTIME(CAST(yyyymm AS VARCHAR), '%Y%m')) AS cal_month,
    CASE 
        WHEN MONTH(STRPTIME(CAST(yyyymm AS VARCHAR), '%Y%m')) >= 10 
        THEN YEAR(STRPTIME(CAST(yyyymm AS VARCHAR), '%Y%m')) + 1
        ELSE YEAR(STRPTIME(CAST(yyyymm AS VARCHAR), '%Y%m')) 
    END AS fiscal_year
FROM distinct_months
ORDER BY 1;
""")
con.execute(f"COPY dim_date TO '{(export_dir / 'dim_date.parquet').as_posix()}' (FORMAT parquet);")
print("Exported: dim_date.parquet")

# 3. Fact: Aggregated Separations
con.execute("""
CREATE OR REPLACE TABLE fact_separations AS
SELECT 
    yyyymm,
    agency_code,
    occ_cat,
    grade_band,
    los_band,
    age_bracket,
    sep_cat,
    drp_flag,
    SUM(n) AS separations
FROM v_sep
GROUP BY ALL;
""")
con.execute(f"COPY fact_separations TO '{(export_dir / 'fact_separations.parquet').as_posix()}' (FORMAT parquet);")
print("Exported: fact_separations.parquet")

# 4. Fact: Aggregated Accessions
con.execute("""
CREATE OR REPLACE TABLE fact_accessions AS
SELECT 
    yyyymm,
    agency_code,
    occ_cat,
    grade_band,
    los_band,
    age_bracket,
    acc_cat,
    SUM(n) AS accessions
FROM v_acc
GROUP BY ALL;
""")
con.execute(f"COPY fact_accessions TO '{(export_dir / 'fact_accessions.parquet').as_posix()}' (FORMAT parquet);")
print("Exported: fact_accessions.parquet")

# 5. Fact: Aggregated Headcount
con.execute(f"COPY fact_headcount TO '{(export_dir / 'fact_headcount.parquet').as_posix()}' (FORMAT parquet);")
print("Exported: fact_headcount.parquet")

print("\n=== WBS 3.1 DELIVERABLE COMPLETE ===")
print("All core dimension and fact Parquet files are exported to the /export folder.")

Exported: dim_agency.parquet
Exported: dim_date.parquet
Exported: fact_separations.parquet
Exported: fact_accessions.parquet
Exported: fact_headcount.parquet

=== WBS 3.1 DELIVERABLE COMPLETE ===
All core dimension and fact Parquet files are exported to the /export folder.


In [38]:
# -------------------------------------------------------------
# WBS 4.2: Comprehensive Dimension Hardening & Uniqueness Enforcement
# -------------------------------------------------------------
export_dir = pathlib.Path("export")
model_dir = pathlib.Path("model_outputs")
export_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)

# 1. Enforce Primary Key Uniqueness on dim_agency
con.execute("""
CREATE OR REPLACE TABLE dim_agency AS
WITH all_agencies AS (
    SELECT agency_code, department_code, agency FROM v_emp WHERE agency_code IS NOT NULL
    UNION
    SELECT agency_code, department_code, agency FROM v_sep WHERE agency_code IS NOT NULL
    UNION
    SELECT agency_code, department_code, agency FROM v_acc WHERE agency_code IS NOT NULL
)
SELECT 
    agency_code,
    FIRST(department_code) AS department_code,
    FIRST(agency)          AS agency
FROM all_agencies
GROUP BY agency_code;
""")
con.execute(f"COPY dim_agency TO '{(export_dir / 'dim_agency.parquet').as_posix()}' (FORMAT parquet);")

# 2. Enforce Primary Key Uniqueness on dim_date
con.execute("""
CREATE OR REPLACE TABLE dim_date AS
WITH distinct_months AS (
    SELECT DISTINCT yyyymm FROM v_sep WHERE yyyymm IS NOT NULL
    UNION
    SELECT DISTINCT yyyymm FROM v_emp WHERE yyyymm IS NOT NULL
    UNION
    SELECT DISTINCT yyyymm FROM v_acc WHERE yyyymm IS NOT NULL
)
SELECT 
    yyyymm,
    STRPTIME(CAST(yyyymm AS VARCHAR), '%Y%m') AS date_ms,
    YEAR(STRPTIME(CAST(yyyymm AS VARCHAR), '%Y%m')) AS cal_year,
    MONTH(STRPTIME(CAST(yyyymm AS VARCHAR), '%Y%m')) AS cal_month,
    CASE 
        WHEN MONTH(STRPTIME(CAST(yyyymm AS VARCHAR), '%Y%m')) >= 10 
        THEN YEAR(STRPTIME(CAST(yyyymm AS VARCHAR), '%Y%m')) + 1
        ELSE YEAR(STRPTIME(CAST(yyyymm AS VARCHAR), '%Y%m')) 
    END AS fiscal_year
FROM distinct_months
ORDER BY 1;
""")
con.execute(f"COPY dim_date TO '{(export_dir / 'dim_date.parquet').as_posix()}' (FORMAT parquet);")

# 3. Enforce Primary Key Uniqueness on dim_agency_archetype
con.execute("""
CREATE OR REPLACE TABLE dim_agency_archetype_clean AS
SELECT 
    agency_code,
    FIRST(agency)              AS agency,
    FIRST(baseline_headcount)  AS baseline_headcount,
    FIRST(drp_exits)           AS drp_exits,
    FIRST(drp_exit_rate)       AS drp_exit_rate,
    FIRST(senior_exit_ratio)   AS senior_exit_ratio,
    FIRST(supervisor_exit_rate)AS supervisor_exit_rate,
    FIRST(cluster_id)          AS cluster_id,
    FIRST(archetype)           AS archetype
FROM read_parquet('model_outputs/dim_agency_archetype.parquet')
GROUP BY agency_code;
""")
con.execute(f"COPY dim_agency_archetype_clean TO '{(model_dir / 'dim_agency_archetype.parquet').as_posix()}' (FORMAT parquet);")

# -------------------------------------------------------------
# Verification: Run validation assertions across all dimensions
# -------------------------------------------------------------
dup_agency = con.sql("SELECT agency_code FROM dim_agency GROUP BY 1 HAVING COUNT(*) > 1").fetchall()
dup_date = con.sql("SELECT yyyymm FROM dim_date GROUP BY 1 HAVING COUNT(*) > 1").fetchall()
dup_arch = con.sql("SELECT agency_code FROM dim_agency_archetype_clean GROUP BY 1 HAVING COUNT(*) > 1").fetchall()

print("=== WBS 4.2: COMPREHENSIVE UNIQUENESS AUDIT ===")
print(f"dim_agency:           {con.sql('SELECT COUNT(*) FROM dim_agency').fetchone()[0]} rows | Duplicates: {len(dup_agency)}")
print(f"dim_date:             {con.sql('SELECT COUNT(*) FROM dim_date').fetchone()[0]} rows | Duplicates: {len(dup_date)}")
print(f"dim_agency_archetype: {con.sql('SELECT COUNT(*) FROM dim_agency_archetype_clean').fetchone()[0]} rows | Duplicates: {len(dup_arch)}")

=== WBS 4.2: COMPREHENSIVE UNIQUENESS AUDIT ===
dim_agency:           133 rows | Duplicates: 0
dim_date:             32 rows | Duplicates: 0
dim_agency_archetype: 49 rows | Duplicates: 0
